# About this notebook
This notebook is for analysing a single image in 2D, to e.g. determine filtering options and visualization.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

import pandas as pd

from iob_ia.utils import filter_labels, visualise
from iob_ia.utils.io_utils import (
    gen_out_path,
    read_image,
    save_image_channel,
    save_labels,
)
from iob_ia.utils.measure import overlapping_labels
from iob_ia.utils.segment import segment_2d_sub_projections
from iob_ia.utils.workflows3d import check_images

### 1.1 Specify a path to an image

In [3]:
# For windows users: use "/" instead of "\" for file paths!
path = "G:/20241120_IOB_Magdalena/20250520_TestClearing/Organoid 2/20250520_10x_Nq146um_405_488_640_01.vsi"

### 1.2 Specify variables

In [4]:
# Channels that should be segmented (0-based indexing)
channels_to_segment = [1,2]
# In case metadata is missing (will be checked later...), specify the voxel size here
voxel_size = None  # or something like: (1.46, 0.65, 0.65)
# Range for substack projections before 2D segmentation
substack_size = 40.0
# To save the images/masks/results, set the following variable to: True
do_save = True

# workflow identifier, used for saving images (no need to change that)
wf = "IOB-2D-workflow_"

### 1.3 Check the image

In [5]:
voxel_size = check_images(image_paths=[path], channels_to_segment=channels_to_segment, voxel_size=voxel_size)

                                    images          channel names  \
0  20250520_10x_Nq146um_405_488_640_01.vsi  [405 C, 488 C, 640 C]   

                                      voxel sizes  
0  (None, 0.6500000000000001, 0.6500000000000001)  


C:\Users\Microscopy\AppData\Local\Temp\ipykernel_33180\4140764759.py:1: UserWarning: 
There are some problems:
- some images do not have proper voxel sizes in the metadata.
-> Please check your images and remove problematic ones from the folder
  voxel_size = check_images(image_paths=[path], channels_to_segment=channels_to_segment, voxel_size=voxel_size)


We got an error...

The image has not proper metadata for voxel sizes -> (**None**, 0.6500, 0.65..).

Since we got an error, we will provide the correct voxel size via a variable (can already be done in "2. Specify variables"). Then re-run the image check.

In [6]:
voxel_size = (1.46, 0.65, 0.65)
voxel_size = check_images(image_paths=[path], channels_to_segment=channels_to_segment, voxel_size=voxel_size)

Using manually defined voxel size.
All good!


---
### 2. Segement the image
Segmentation is done in sub-parts of the stack, i.e. projections of parts of the stack, then recombining the projections to a 3D stack.

In [7]:
# Open the image
image = read_image(path)
# Load the image data
img = image.get_data()

# We will keep the results in memory
channel_images = []
cp_masks = []
# Segment the channels
for ch in channels_to_segment:
    cp_mask, channel, mod_voxel_size = segment_2d_sub_projections(img[ch], voxel_size=voxel_size, substack_size=substack_size)
    channel_images.append(channel)
    cp_masks.append(cp_mask)
    # Optionally save results
    if do_save:
        cp_mask_path = gen_out_path(path, wf + f'cp-mask_ch{ch}-{image.channel_names[ch]}')
        channel_path = gen_out_path(path, wf + f'image_ch{ch}-{image.channel_names[ch]}.ome')
        save_labels(cp_mask, cp_mask_path)
        save_image_channel(channel, channel_path, voxel_size=mod_voxel_size)



Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.6 
platform:       	win32 
python version: 	3.11.13 
torch version:  	2.8.0+cu126! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 


2025-11-03 10:33:09,742 [INFO] WRITING LOG OUTPUT TO C:\Users\Microscopy\.cellpose\run.log
2025-11-03 10:33:09,743 [INFO] 
cellpose version: 	4.0.6 
platform:       	win32 
python version: 	3.11.13 
torch version:  	2.8.0+cu126
>>> GPU activated: True
2025-11-03 10:33:09,745 [INFO] ** TORCH CUDA version installed and working. **
2025-11-03 10:33:09,745 [INFO] >>>> using GPU (CUDA)
2025-11-03 10:33:11,085 [INFO] >>>> loading model C:\Users\Microscopy\.cellpose\models\cpsam
Saved labels to: G:/20241120_IOB_Magdalena/20250520_TestClearing/Organoid 2\20250520_10x_Nq146um_405_488_640_01_IOB-2D-workflow_cp-mask_ch1-488 C.tif
Saved image to: G:/20241120_IOB_Magdalena/20250520_TestClearing/Organoid 

---
### 3. Visualising and finding your filtering options

In [8]:
# We will visualise the sub-stack projected raw channels and their segmentations
for i, ch in enumerate(channels_to_segment):
    visualise.add_image(channel_images[i], name="raw_"+image.channel_names[ch], scale=mod_voxel_size)
    visualise.add_labels(cp_masks[i], name="cp-mask_"+image.channel_names[ch], scale=mod_voxel_size)

In the napari viewer that opened, you can see the image and also toggle to 3D view if you like.

Under `Plugins` you can find the `Filter labels by properties` plugin. With this plugin you can interactively filter the cellpose objects by some properties (e.g. area (which is the volume in um3), or channel intensities).

Take note of what property you want to filter on and the values.

We will not consider the volume (area) as this takes the z-axis into account. Better to use the `projected_area` for these 2D segmentations.
- `projected_area` = 42-233 um2
And we decide to filter on intensity:
- channel 1 (GFP) `intensity_mean`: "mean/3" - infinity
- channel 2 (RFP) `intensity_mean`: 150 - infinity
- channel 2 (RFP) `intensity_max`: 0 - 15000

**Absolute intensity values** are generally a bad idea for (later) batch analysis. Hence, I already "introduced" a dynamic minimal intensity_mean value for the GFP channel, calculated as the average of all objects GFP intensity mean divided by 3.
This should be check on different images...

---
## 4. Filtering label masks
First we will measure the properties of the label masks. Using `verbose=True` allows us to print out some information that could be useful.

In [9]:
properties = []
for i in range(len(cp_masks)):
    print("--> Measuring properties for channel:", image.channel_names[channels_to_segment[i]])
    props = filter_labels.measure_label_props(
        img_label=cp_masks[i],
        img_intensity=channel_images[i],
        voxel_size=mod_voxel_size,
        verbose=True
    )
    properties.append(props)

--> Measuring properties for channel: 488 C
Measured properties of 6806 labels.
area: mean = 3638.722; median = 3515.2; bimodal = 3983.413
euler_number: mean = 1.0; median = 1.0; bimodal = nan
extent: mean = 0.746; median = 0.754; bimodal = 0.746
intensity_mean: mean = 640.95; median = 516.202; bimodal = 933.348
intensity_min: mean = 274.388; median = 246.0; bimodal = 330.931
intensity_max: mean = 1080.45; median = 812.0; bimodal = 1739.801
projected_area: mean = 90.968; median = 87.88; bimodal = 99.036
projected_convex_area: mean = 95.391; median = 92.105; bimodal = 104.625
projected_perimeter: mean = 34.439; median = 34.05; bimodal = 34.445
projected_circularity: mean = 0.941; median = 0.952; bimodal = 0.941
--> Measuring properties for channel: 640 C


C:\Users\Microscopy\.conda\envs\iob-ia\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Users\Microscopy\.conda\envs\iob-ia\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Measured properties of 7801 labels.
area: mean = 2750.031; median = 2602.6; bimodal = 3327.935
euler_number: mean = 1.002; median = 1.0; bimodal = nan
extent: mean = 0.746; median = 0.753; bimodal = 0.746
intensity_mean: mean = 461.255; median = 436.086; bimodal = 508.692
intensity_min: mean = 243.005; median = 234.0; bimodal = 262.793
intensity_max: mean = 711.261; median = 657.0; bimodal = 804.653
projected_area: mean = 68.751; median = 65.065; bimodal = 82.582
projected_convex_area: mean = 72.63; median = 68.445; bimodal = 88.903
projected_perimeter: mean = 29.478; median = 29.008; bimodal = 29.484
projected_circularity: mean = 0.945; median = 0.956; bimodal = 0.945


We will use dictionaries to define properties and ranges for them to filter objects.

For values calculated dynamically per image we can use strings like `mean`, `median`, or `bimodal` (the latter uses the the `estimate_threshold` function).
We can in addition use divisions or multiplication of those values like:
- "median/4"
- "bimodal*2.3"
- "mean/3"

In [10]:
filter_dictionary = [
    # For the first channel (GFP)
    {
        "projected_area": [42, 233],
        "intensity_mean": ["mean/3", float("inf")],
    },
    # For the second channel (RFP)
    {
        "projected_area": [42, 233],
        "intensity_mean": [150, None],
        "intensity_max": [float("-inf"), 15000]
    }
]

In [11]:
# Filter the cellpose masks
filtered_masks = []
filtered_properties = []

for i in range(len(cp_masks)):
    print("--> Filtering cellpose mask for channel:", image.channel_names[channels_to_segment[i]])
    mask, updated_props = filter_labels.filter_labels_by_property(
        img_label=cp_masks[i],
        props_table=properties[i],
        props=filter_dictionary[i],
        verbose=True
    )
    filtered_masks.append(mask)
    filtered_properties.append(updated_props)
    if do_save:
        mask_path = gen_out_path(path, wf + f'filtered-mask_ch{channels_to_segment[i]}-{image.channel_names[channels_to_segment[i]]}')
        save_labels(labels=mask, path=mask_path)

--> Filtering cellpose mask for channel: 488 C
Filtering on projected_area between 42 and 233
Removed 158 labels. Remaining labels: 6648 (previously 6806)
Filtering on intensity_mean between mean/3 and inf
Removed 369 labels. Remaining labels: 6437 (previously 6806)
Creating filtered mask...
Final filtered image has 6299 labels.
Saved labels to: G:/20241120_IOB_Magdalena/20250520_TestClearing/Organoid 2\20250520_10x_Nq146um_405_488_640_01_IOB-2D-workflow_filtered-mask_ch1-488 C.tif
--> Filtering cellpose mask for channel: 640 C
Filtering on projected_area between 42 and 233
Removed 898 labels. Remaining labels: 6903 (previously 7801)
Filtering on intensity_mean between 150 and inf
Removed 153 labels. Remaining labels: 7648 (previously 7801)
Filtering on intensity_max between -inf and 15000
Removed 0 labels. Remaining labels: 7801 (previously 7801)
Creating filtered mask...
Final filtered image has 6788 labels.
Saved labels to: G:/20241120_IOB_Magdalena/20250520_TestClearing/Organoid 2\

---
### 5. Counting double positive objects
The approach I used here is as follows: if an object in channel 488 overlap at least by 50% with object**s** in channel 640, they are considered as a `640 positive 488 object(s)`.

We can create masks accordingly. The fuction below will do that and takes:
- first mask = objects of interest
- second mask = selector mask (to check if the first mask objects have overlap with the second mask)
- `min_overlap`: if not included here as a parameter, it defaults to 0.5 (=50%). But can be changed...

In [12]:
dp_masks = []
if len(filtered_masks) != 2:
    print("Counting double positive objects is only implemented for segmenting exactly 2 channels.")
else:
    dp_mask_1 = overlapping_labels(filtered_masks[0], filtered_masks[1], min_overlap=0.5)
    dp_mask_2 = overlapping_labels(filtered_masks[1], filtered_masks[0], min_overlap=0.5)
    dp_masks.append(dp_mask_1)
    dp_masks.append(dp_mask_2)
    if do_save:
        out_path = gen_out_path(path, wf + f'mask_{image.channel_names[channels_to_segment[1]]}-pos-{image.channel_names[channels_to_segment[0]]}-objects')
        save_labels(dp_mask_1, out_path)
        out_path = gen_out_path(path, wf + f'mask_{image.channel_names[channels_to_segment[0]]}-pos-{image.channel_names[channels_to_segment[1]]}-objects')
        save_labels(dp_mask_2, out_path)

Saved labels to: G:/20241120_IOB_Magdalena/20250520_TestClearing/Organoid 2\20250520_10x_Nq146um_405_488_640_01_IOB-2D-workflow_mask_640 C-pos-488 C-objects.tif
Saved labels to: G:/20241120_IOB_Magdalena/20250520_TestClearing/Organoid 2\20250520_10x_Nq146um_405_488_640_01_IOB-2D-workflow_mask_488 C-pos-640 C-objects.tif


---
## 6. Count all objects and create result table

In [13]:
# Initialise the result table
result_table = {
    "Image-Path": [path],
    "Image-Name": [os.path.basename(path)],
}
for ch in channels_to_segment:
    result_table[f'cp-count-ch{ch}'] = []
for ch in channels_to_segment:
    result_table[f'object-count-ch{ch}'] = []

# Count objects
for i in range(len(channels_to_segment)):
    cp_count = filter_labels.count_unique_labels(cp_masks[i])
    result_table[f'cp-count-ch{channels_to_segment[i]}'].append(cp_count)
    object_count = filter_labels.count_unique_labels(filtered_masks[i])
    result_table[f'object-count-ch{channels_to_segment[i]}'].append(object_count)
if dp_masks:
    result_table[f'ch{channels_to_segment[1]}pos-ch{channels_to_segment[0]}objects'] = [
        filter_labels.count_unique_labels(dp_masks[0])
    ]
    result_table[f'ch{channels_to_segment[0]}pos-ch{channels_to_segment[1]}objects'] = [
        filter_labels.count_unique_labels(dp_masks[1])
    ]
result_table = pd.DataFrame.from_dict(result_table)
if do_save:
    out_path = path + wf + 'single_result_table.csv'
    pd.DataFrame.to_csv(result_table, out_path)
    print("Saved result table to:", out_path)
result_table

Saved result table to: G:/20241120_IOB_Magdalena/20250520_TestClearing/Organoid 2/20250520_10x_Nq146um_405_488_640_01.vsiIOB-2D-workflow_single_result_table.csv


,Image-Path,Image-Name,cp-count-ch1,cp-count-ch2,object-count-ch1,object-count-ch2,ch2pos-ch1objects,ch1pos-ch2objects
0,G:/20241120_IOB_Magdalena/20250520_TestClearin...,20250520_10x_Nq146um_405_488_640_01.vsi,6806,7801,6299,6788,2809,3101


---
## 7. Visualising the results
Assuming the napari window was not closed, the following cell will add the remaining results to the napari viewer.

In [14]:
# The following line would add the original multichannel image (all channels and in color)
visualise.add_multichannel_image(img, name="raw", channel_names=image.channel_names, scale=voxel_size)
for i, ch in enumerate(channels_to_segment):
    # The next two lines would add the same images as already done in Step 3
    #visualise.add_image(channel_images[i], name="raw_"+image.channel_names[ch], scale=mod_voxel_size)
    #visualise.add_labels(cp_masks[i], name="cp-mask_"+image.channel_names[ch], scale=mod_voxel_size)
    # Add the filtered masks
    visualise.add_labels(filtered_masks[i], name="filtered_mask"+image.channel_names[ch], scale=mod_voxel_size)
# Add the double postivie masks
if dp_masks:
    visualise.add_labels(
        dp_masks[0],
        name=f'ch{image.channel_names[channels_to_segment[1]]}pos-ch{image.channel_names[channels_to_segment[0]]}-objects_mask',
        scale=mod_voxel_size
    )
    visualise.add_labels(
        dp_masks[1],
        name=f'ch{image.channel_names[channels_to_segment[0]]}pos-ch{image.channel_names[channels_to_segment[1]]}-objects_mask',
        scale=mod_voxel_size)